# 02 - Preprocessing & Baseline

---

Este notebook implementa:
1. Carga y exploración del dataset
2. Feature engineering (date features, investor count, valuation cleaning)
3. Definición de X e y (target: valuation_b)
4. Pipeline de preprocesamiento (imputación, escalado, OneHotEncoder)
5. Entrenamiento de baselines: DummyRegressor, LinearRegression, Ridge
6. Evaluación con MAE, MSE, RMSE, R2
7. Comparación train/test para detectar overfitting
8. Guardado del modelo con joblib

In [1]:
import sys
import os
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')

# Ajustar path para imports desde la raíz del proyecto
sys.path.insert(0, os.path.abspath('..'))

from src.preprocessing.preprocessing_pipeline import (
    FeatureEngineer,
    get_feature_types,
    build_preprocessor,
    save_clean_data,
)
from src.models.evaluate import compare_train_test, detect_overfitting, print_evaluation_report

print('Librerías cargadas correctamente.')

Librerías cargadas correctamente.


## 1. Carga del Dataset

---

In [2]:
import kagglehub

path = kagglehub.dataset_download('ramjasmaurya/unicorn-startups')
csv_file = os.path.join(path, 'unicorns till sep 2022.csv')
df = pd.read_csv(csv_file)

print(f'Shape: {df.shape}')
df.head()

Shape: (1186, 7)


,Company,Valuation ($B),Date Joined,Country,City,Industry,Investors
0,ByteDance,$140,4/7/2017,China,Beijing,Artificial intelligence,"Sequoia Capital China, SIG Asia Investments, S..."
1,SpaceX,$127,12/1/2012,United States,Hawthorne,Other,"Founders Fund, Draper Fisher Jurvetson, Rothen..."
2,SHEIN,$100,7/3/2018,China,Shenzhen,E-commerce & direct-to-consumer,"Tiger Global Management, Sequoia Capital China..."
3,Stripe,$95,1/23/2014,United States,San Francisco,Fintech,"Khosla Ventures, LowercaseCapital, capitalG"
4,Canva,$40,1/8/2018,Australia,Surry Hills,Internet software & services,"Sequoia Capital China, Blackbird Ventures, Mat..."


In [3]:
print('Columnas:', df.columns.tolist())
print()
print(df.info())
print()
print('Valores nulos:')
print(df.isnull().sum())

Columnas: ['Company', 'Valuation ($B)', 'Date Joined', 'Country', 'City\xa0', 'Industry', 'Investors']

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1186 entries, 0 to 1185
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Company         1186 non-null   object
 1   Valuation ($B)  1186 non-null   object
 2   Date Joined     1186 non-null   object
 3   Country         1186 non-null   object
 4   City            1186 non-null   object
 5   Industry        1186 non-null   object
 6   Investors       1168 non-null   object
dtypes: object(7)
memory usage: 65.0+ KB
None

Valores nulos:
Company            0
Valuation ($B)     0
Date Joined        0
Country            0
City               0
Industry           0
Investors         18
dtype: int64


## 2. Feature Engineering

---

In [4]:
fe = FeatureEngineer(reference_date='2022-09-01')
df_eng = fe.fit_transform(df)

print(f'Después de feature engineering: {df_eng.shape}')
print(f'Columnas: {df_eng.columns.tolist()}')
df_eng.head()

Después de feature engineering: (1186, 8)
Columnas: ['Country', 'City', 'Industry', 'join_year', 'join_month', 'years_since_joined', 'investors_count', 'valuation_b']


,Country,City,Industry,join_year,join_month,years_since_joined,investors_count,valuation_b
0,China,Beijing,Artificial intelligence,2017,4,5.40,4,140.0
1,United States,Hawthorne,Other,2012,12,9.75,3,127.0
2,China,Shenzhen,E-commerce & direct-to-consumer,2018,7,4.16,3,100.0
3,United States,San Francisco,Fintech,2014,1,8.61,3,95.0
4,Australia,Surry Hills,Internet software & services,2018,1,4.65,3,40.0


In [5]:
print('Tipos de datos:')
print(df_eng.dtypes)
print()
print('Estadísticas descriptivas:')
df_eng.describe()

Tipos de datos:
Country                object
City                   object
Industry               object
join_year               Int64
join_month              Int64
years_since_joined    float64
investors_count         int64
valuation_b           float64
dtype: object

Estadísticas descriptivas:


,join_year,join_month,years_since_joined,investors_count,valuation_b
count,1186.0,1186.0,1186.000000,1186.000000,1186.000000
mean,2020.12226,6.242833,2.067926,2.801855,3.251282
std,1.984172,3.402415,1.946342,0.592809,7.641574
min,2007.0,1.0,0.020000,0.000000,1.000000
25%,2019.0,3.0,0.792500,3.000000,1.100000
50%,2021.0,6.0,1.315000,3.000000,1.600000
75%,2021.0,9.0,2.967500,3.000000,3.000000
max,2022.0,12.0,15.170000,4.000000,140.000000


## 2b. Guardado del Dataset Limpio

---

In [6]:
data_dir = '../data'
os.makedirs(data_dir, exist_ok=True)
clean_path = os.path.join(data_dir, 'dataset_clean.csv')
df_eng.to_csv(clean_path, index=False)
print(f'Dataset limpio guardado en: {clean_path}')
print(f'Shape: {df_eng.shape}')
print(f'Columnas: {df_eng.columns.tolist()}')

Dataset limpio guardado en: ../data\dataset_clean.csv
Shape: (1186, 8)
Columnas: ['Country', 'City', 'Industry', 'join_year', 'join_month', 'years_since_joined', 'investors_count', 'valuation_b']


## 3. Definición de X e y

---

In [7]:
y = df_eng['valuation_b']
X = df_eng.drop(columns=['valuation_b'])

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Features: {X.columns.tolist()}')
print(f'Target: valuation_b (min={y.min():.2f}, max={y.max():.2f}, mean={y.mean():.2f})')

X shape: (1186, 7)
y shape: (1186,)
Features: ['Country', 'City', 'Industry', 'join_year', 'join_month', 'years_since_joined', 'investors_count']
Target: valuation_b (min=1.00, max=140.00, mean=3.25)


## 4. Train/Test Split

---

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {X_train.shape[0]} muestras')
print(f'Test:  {X_test.shape[0]} muestras')

Train: 948 muestras
Test:  238 muestras


## 5. Pipeline de Preprocesamiento

---

In [9]:
numeric_cols, categorical_cols = get_feature_types(X)
print(f'Numéricas: {numeric_cols}')
print(f'Categóricas: {categorical_cols}')

from sklearn.pipeline import Pipeline
preprocessor = build_preprocessor(numeric_cols, categorical_cols)
pipeline = Pipeline(steps=[('preprocessor', preprocessor)])
pipeline.fit(X_train, y_train)

X_train_transformed = pipeline.transform(X_train)
X_test_transformed = pipeline.transform(X_test)

print(f'\nX_train transformado: {X_train_transformed.shape}')
print(f'X_test transformado:  {X_test_transformed.shape}')
print(f'Valores nulos en train: {np.isnan(X_train_transformed).sum()}')
print(f'Valores nulos en test:  {np.isnan(X_test_transformed).sum()}')

Numéricas: ['join_year', 'join_month', 'years_since_joined', 'investors_count']
Categóricas: ['Country', 'Industry']

X_train transformado: (948, 80)
X_test transformado:  (238, 80)
Valores nulos en train: 0
Valores nulos en test:  0


## 6. Entrenamiento de Baselines

---

In [10]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models = {
    'DummyRegressor': DummyRegressor(strategy='mean'),
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
}

results = {}

for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', pipeline),
        ('model', model),
    ])
    pipe.fit(X_train, y_train)

    y_pred_train = pipe.predict(X_train)
    y_pred_test = pipe.predict(X_test)

    train_metrics = {
        'MAE': mean_absolute_error(y_train, y_pred_train),
        'MSE': mean_squared_error(y_train, y_pred_train),
        'RMSE': np.sqrt(mean_squared_error(y_train, y_pred_train)),
        'R2': r2_score(y_train, y_pred_train),
    }
    test_metrics = {
        'MAE': mean_absolute_error(y_test, y_pred_test),
        'MSE': mean_squared_error(y_test, y_pred_test),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test)),
        'R2': r2_score(y_test, y_pred_test),
    }

    results[name] = {
        'pipeline': pipe,
        'train': train_metrics,
        'test': test_metrics,
    }

print('Entrenamiento completado.')

Entrenamiento completado.


## 7. Evaluación y Comparación

---

In [11]:
report_df = print_evaluation_report(results)


EVALUATION REPORT
           model split      MAE       MSE     RMSE        R2
  DummyRegressor train 2.857393 70.683979 8.407376  0.000000
  DummyRegressor  test 2.109304  9.295632 3.048874 -0.057875
LinearRegression train 2.952648 62.895022 7.930638  0.110194
LinearRegression  test 2.479947 16.449496 4.055798 -0.872009
           Ridge train 2.902053 63.179377 7.948546  0.106171
           Ridge  test 2.440321 16.051468 4.006428 -0.826712

------------------------------------------------------------

WARNINGS:
  - UNDERFITTING WARNING: DummyRegressor | Test R2=-0.0579 < 0
  - OVERFITTING WARNING: LinearRegression | Train R2=0.1102, Test R2=-0.8720, Gap=0.9822
  - UNDERFITTING WARNING: LinearRegression | Test R2=-0.8720 < 0
  - OVERFITTING WARNING: Ridge | Train R2=0.1062, Test R2=-0.8267, Gap=0.9329
  - UNDERFITTING WARNING: Ridge | Test R2=-0.8267 < 0


In [12]:
for name, res in results.items():
    print(f'\n--- {name} ---')
    print(f'  Train R2: {res["train"]["R2"]:.4f}')
    print(f'  Test  R2: {res["test"]["R2"]:.4f}')
    gap = res['train']['R2'] - res['test']['R2']
    print(f'  Gap R2:   {gap:.4f}')
    if gap > 0.15:
        print(f'  ** Posible OVERFITTING **')
    elif res['test']['R2'] < 0:
        print(f'  ** Posible UNDERFITTING **')
    else:
        print(f'  OK')


--- DummyRegressor ---
  Train R2: 0.0000
  Test  R2: -0.0579
  Gap R2:   0.0579
  ** Posible UNDERFITTING **

--- LinearRegression ---
  Train R2: 0.1102
  Test  R2: -0.8720
  Gap R2:   0.9822
  ** Posible OVERFITTING **

--- Ridge ---
  Train R2: 0.1062
  Test  R2: -0.8267
  Gap R2:   0.9329
  ** Posible OVERFITTING **


## 8. Guardado del Modelo

---

In [13]:
best_name = min(results, key=lambda k: results[k]['test']['RMSE'])
best_pipeline = results[best_name]['pipeline']

model_dir = '../models'
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, 'best_model.joblib')
joblib.dump(best_pipeline, model_path)

print(f'Mejor modelo: {best_name}')
print(f'Guardado en: {model_path}')

# Verificar que se puede cargar
loaded = joblib.load(model_path)
test_pred = loaded.predict(X_test.head(1))
print(f'Predicción de prueba: {test_pred[0]:.2f}B')

Mejor modelo: DummyRegressor
Guardado en: ../models\best_model.joblib
Predicción de prueba: 3.39B


## 9. Verificación de No Data Leakage

---

El pipeline está diseñado para evitar data leakage:
- Feature engineering se aplica a todos los datos (no usa información del test)
- El preprocessor (imputación, escalado, OHE) se fittea SOLO con train
- Test se transforma con los parámetros aprendidos del train

In [14]:
print('Verificación de no data leakage:')
print(f'1. FeatureEngineer se fittea en train y transforma test (sin leakage)')
print(f'2. Preprocessor (imputer, scaler, encoder) se fittea solo en train')
print(f'3. Test se transforma con los parámetros del train')
print(f'4. Pipeline completo: FeatureEngineering -> Preprocessor -> Model')
print(f'\nEl modelo guardado contiene todo el pipeline y puede usarse directamente.')

Verificación de no data leakage:
1. FeatureEngineer se fittea en train y transforma test (sin leakage)
2. Preprocessor (imputer, scaler, encoder) se fittea solo en train
3. Test se transforma con los parámetros del train
4. Pipeline completo: FeatureEngineering -> Preprocessor -> Model

El modelo guardado contiene todo el pipeline y puede usarse directamente.
